In [ ]:
import pandas as pd

In [ ]:
pd.read_parquet("../artifacts/augmented/train_meta_aug.parquet")

In [ ]:
pd.read_parquet("../artifacts/augmented/train_obs_aug.parquet")

In [ ]:
pd.read_parquet("../artifacts/features/train_feats.parquet")

In [ ]:
pd.read_parquet("../artifacts/features/test_feats.parquet")

In [ ]:
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

# Connect to the EXISTING database and study
# (Do not create a new study, just load it)
study = optuna.load_study(
    study_name="cb_study_20251216-091312-+0000", 
    storage="sqlite:///D:/root/dev/py/mallorn/optuna.db"
)

# These will auto-update every time you re-run the cell
print(f"Current Trials: {len(study.trials)}")
plot_optimization_history(study)

In [ ]:
plot_param_importances(study)

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from mallorn.common import SEED

# 2. Prepare your data (Make sure X_train/y_train are loaded in your notebook)
feats_df = pd.read_parquet("../artifacts/features/train_feats.parquet")
X_train = feats_df.drop(columns=["target"])
y_train = feats_df["target"]

n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos = n_neg / n_pos if n_pos > 0 else 1.0

# 1. Load the Best Parameters from your fixed study
best_params = study.best_params
best_params.update({
    # "task_type": "GPU",
    "loss_function": "Logloss",
    "scale_pos_weight": scale_pos,
    "eval_metric": "AUC",
    "verbose": False,
    "random_seed": SEED,
    "allow_writing_files": False,
})
print(f"Retraining model with best params: {best_params}")

# 3. Retrain the model
# Note: You might need to manually add fixed params that weren't tuned 
# (e.g., iterations, loss_function) if they aren't in best_params.
model = CatBoostClassifier(
    **best_params,
)
model.fit(X_train, y_train)

# 4. Get Feature Importance
importance = model.get_feature_importance(type='FeatureImportance', prettified=True)

# 5. Plotting (Simple Bar Chart)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.barplot(x="Importances", y="Feature Id", data=importance.head(20)) # Top 20 features
plt.title('CatBoost Feature Importance')
plt.show()